In [1]:
from pyspark.sql import SparkSession

# packages for iceberg + spark + kafka
PACKAGES = [
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3",
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "org.postgresql:postgresql:42.6.0",
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"
]

spark = SparkSession.builder \
    .appName("Iceberg_Setup") \
    .config("spark.jars.packages", ",".join(PACKAGES)) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.catalog-impl", "org.apache.iceberg.jdbc.JdbcCatalog") \
    .config("spark.sql.catalog.my_catalog.uri", "jdbc:postgresql://postgres:5432/iceberg_metastore") \
    .config("spark.sql.catalog.my_catalog.jdbc.user", "iceberg") \
    .config("spark.sql.catalog.my_catalog.jdbc.password", "iceberg") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://warehouse/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()
print('123')

123


In [2]:
# create default namespace and table
spark.sql("CREATE NAMESPACE IF NOT EXISTS my_catalog.default")

spark.sql("""
CREATE TABLE IF NOT EXISTS my_catalog.default.user_events (
    user_id INT,
    shop STRING,    
    event_type STRING,
    device_type STRING,
    price DOUBLE,
    session_id STRING,
    timestamp TIMESTAMP
)
USING iceberg
TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet'
)
""")

print("table created succesfully")

spark.sql("DESCRIBE my_catalog.default.user_events").show()

table created succesfully
+-----------+---------+-------+
|   col_name|data_type|comment|
+-----------+---------+-------+
|    user_id|      int|   NULL|
|       shop|   string|   NULL|
| event_type|   string|   NULL|
|device_type|   string|   NULL|
|      price|   double|   NULL|
| session_id|   string|   NULL|
|  timestamp|timestamp|   NULL|
+-----------+---------+-------+



In [4]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType
from pyspark.sql.functions import col, from_json, cast

# base streaming config for kafka

json_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("shop", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("session_id", StringType(), True),
    StructField("timestamp", TimestampType(), True)
])

print('123')

# read stream in bytes
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "user_events") \
    .option("startingOffsets", "latest") \
    .load()

print('456')

# parse bytes to string and enforce schema
parsed_stream = kafka_stream \
    .selectExpr("CAST(value AS STRING) as json_string") \
    .select(from_json(col("json_string"), json_schema).alias("data")) \
    .select("data.*")

print('789')

# write data into files 
streaming_query = parsed_stream.writeStream \
    .format("iceberg") \
    .outputMode("append") \
    .trigger(processingTime="3 seconds") \
    .option("checkpointLocation", "s3a://warehouse/checkpoints/user_events_v1") \
    .toTable("my_catalog.default.user_events")

123
456
789


In [5]:
spark.sql("SELECT * FROM my_catalog.default.user_events ORDER BY timestamp DESC LIMIT 5").show()

+-------+-------+-----------+-----------+-----+--------------------+--------------------+
|user_id|   shop| event_type|device_type|price|          session_id|           timestamp|
+-------+-------+-----------+-----------+-----+--------------------+--------------------+
|   9727|foxtrot|   purchase|    desktop|  0.0|26ca3433-6022-4fd...|2026-05-20 20:23:...|
|   4848|foxtrot|change_lang|    desktop|  0.0|449f354a-c19f-46f...|2026-05-20 20:23:...|
|    216|foxtrot|  page_view|    android|69.55|2c749995-2912-4cf...|2026-05-20 20:23:...|
|   4021|dniprom|  page_view|    desktop|  0.0|26f6b798-7114-418...|2026-05-20 20:23:...|
|   5893|dniprom|   purchase|        ios|  0.0|ee3087ae-1c2a-4e9...|2026-05-20 20:23:...|
+-------+-------+-----------+-----------+-----+--------------------+--------------------+



In [12]:
spark.sql("DROP TABLE my_catalog.default.user_events")

DataFrame[]

In [7]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType
from pyspark.sql.functions import col, from_json

json_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("shop", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("session_id", StringType(), True),
    StructField("timestamp", TimestampType(), True)
])


kafka_stream_sorted = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "user_events") \
    .option("startingOffsets", "latest") \
    .load()

parsed_stream_sorted = kafka_stream_sorted \
    .selectExpr("CAST(value AS STRING) as json_string") \
    .select(from_json(col("json_string"), json_schema).alias("data")) \
    .select("data.*")


streaming_query_sorted = parsed_stream_sorted.writeStream \
    .format("iceberg") \
    .outputMode("append") \
    .trigger(processingTime="3 seconds") \
    .option("checkpointLocation", "s3a://warehouse/checkpoints/sorted_v1") \
    .toTable("my_catalog.default.user_events_sorted")
print("12312")

12312
